# Main notebook

### Init

In [1]:
import torch
import torch_geometric
import dgl
import deepchem
import rdkit

print(f"PyTorch:       {torch.__version__}")
print(f"PyG:           {torch_geometric.__version__}")
print(f"DGL:           {dgl.__version__}")
print(f"DeepChem:      {deepchem.__version__}")
print(f"RDKit:         {rdkit.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}") 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/vollmers/.conda/envs/gnn-m/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/vollmers/.conda/envs/gnn-m/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/vollmers/.conda/envs/gnn-m/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/vollmers/.conda/envs/gnn-m/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in l

PyTorch:       2.2.1+cpu
PyG:           2.7.0
DGL:           2.1.0
DeepChem:      2.8.0
RDKit:         2025.09.6
CUDA available: False


In [3]:
# git add .
# git commit -m "message"
# git push origin main


%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.data.data import load_data, print_mol_types
from src.utils.plotting import plot_smiles, plot_metals, plot_training
from src.data.preprocess import keep_largest, salt_remover, preprocess
from src.model.eval import train

from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import AllChem, Draw, PandasTools, Descriptors
from rdkit.Chem.rdmolops import GetAdjacencyMatrix
from rdkit.Chem.Draw import IPythonConsole

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Init data

In [4]:
path = Path("Data") / "toxicity_fish.csv"
selected_columns = ["SMILES", "conc"]
cut = 200
# cut = None

df = load_data(path, selected_columns, cut)

print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'Data/toxicity_fish.csv'

### Data analysis

In [4]:
# Unique smiles
# plot_smiles(df, figsize=[12, 6])

In [5]:
# Mol types and metals

mol_id = 16
print(df.head())
mol_types(df)
# plot_metals(df)
print(df['SMILES'][199])

       SMILES  conc
0  O=[O+][O-]  0.18
1  O=[O+][O-]  0.18
2  O=[O+][O-]  0.26
3  O=[O+][O-]  0.26
4  O=[O+][O-]  0.17
Total molecules: 200
Unique molecules: 29
Salts: 83, 41.50%
Single atoms: 6, 3.00%
Metals: 65, 32.50%
[Cl-].[NH4+]


### Data processing

In [6]:
# Preprocess by splitting salts, removing lone atoms and removing metals, also converting concentrations to log scale
df_pp = preprocess(df)

print(df_pp.head())
mol_types(df_pp)

       SMILES      conc
0  O=[O+][O-] -0.744727
1  O=[O+][O-] -0.744727
2  O=[O+][O-] -0.585027
3  O=[O+][O-] -0.585027
4  O=[O+][O-] -0.769551
Total molecules: 145
Unique molecules: 24
Salts: 0, 0.00%
Single atoms: 0, 0.00%
Metals: 0, 0.00%


### DeepChem

In [7]:
import deepchem as dc

# Convert to deepchem
featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
smiles_dc = featurizer.featurize(df_pp['SMILES'].tolist())

Failed to featurize datapoint 143, [NH4+]. Appending empty array
Exception message: More than one atom should be present in the molecule for this featurizer to work.
Failed to featurize datapoint 144, [NH4+]. Appending empty array
Exception message: More than one atom should be present in the molecule for this featurizer to work.
Exception message: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (145,) + inhomogeneous part.


In [8]:
mol_id = 16

print(type(smiles_dc[mol_id]))

print(smiles_dc[mol_id])
print("")
# node_features: n_atoms x n_node_features (feature length = 30)
# - Atom type, formal charge, hybridization, hydrogen bonding, aromatic, degree, n_hydrogens, chirality, partial charge
print("Node features for one atom example:")
print(smiles_dc[mol_id].node_features[2])
# edge_index: 2 x n_edges (source and target) - shows bonds
# edge_features: n_edges x n_edge_features (feature length = 11)
# - Bond type, same ring, conjugated, stereo
print("Edge features for one bond example:")
print(smiles_dc[mol_id].edge_features[2])

<class 'deepchem.feat.graph_data.GraphData'>
GraphData(node_features=[9, 30], edge_index=[2, 18], edge_features=[18, 11])

Node features for one atom example:
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.
 0. 1. 0. 0. 0. 0.]
Edge features for one bond example:
[0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0.]


In [9]:
# Create a dataset
dataset = dc.data.NumpyDataset(X=smiles_dc, y=df_pp['conc'].values, ids=df_pp['SMILES'].tolist())

# print(dataset)

# Split dataset
splitter = dc.splits.RandomSplitter() # Random split, maybe use some other splitter later
train_dataset, test_dataset = splitter.train_test_split(dataset, seed=42)

print(f"Train set size: {len(train_dataset)} - {len(train_dataset) / len(dataset) * 100:.2f}%")
print(f"Test set size: {len(test_dataset)} - {len(test_dataset) / len(dataset) * 100:.2f}%")

Train set size: 116 - 80.00%
Test set size: 29 - 20.00%


# GNN

In [10]:
from deepchem.models.torch_models import GCNModel
import dgl
import torch

# GCNModel for regression (predicting concentration)
model = GCNModel(
    mode='classification', 
    n_tasks=1, 
    batch_size=16, 
    learning_rate=0.001,
    number_atom_features=30,
    device=torch.device('cpu')
)

In [11]:
model.fit(train_dataset, nb_epoch=10)

: 